# Baselines on stereo-stripped + 2D-InChIKey-unique S4 candidates (512-cap)

Retrieval evaluation on the S4-augmented retrieval candidates after three
fixes to the pool:

1. **Stereo-stripping** of both queries and candidates (removes the
   S4-vs-database asymmetry shortcut — S4's ChEMBL31 vocab has no stereo
   tokens).
2. **Per-query 2D-InChIKey dedup applied before the 1024→512 trim**, so
   every candidate list is guaranteed unique by 2D-InChIKey (no tautomers
   sharing the query's 2D structure leak in as decoys).
3. **Cap reduced to 512** (vs 1024) — random hit@1 doubles, and per-batch
   fingerprinting time halves.

All numbers are per spectrum (n ≈ 17.5k test spectra after a small TSV
re-alignment). Bootstrap CIs over spectra. Two tables: mass (S4+PubChem
fallback, ±10 ppm) and formula (S4+PubChem+Molpher).

**Methods**
- **Random** — per-spectrum theoretical baseline, averaged over 100 draws.
- **Chirality** — RDKit chiral-atom count + train-fitted direction.
- **ChemBERTa-77M-MLM binary** — per-candidate classifier on
  (query=pos, cand=neg) pairs.
- **DeepSets / DeepSets+FF / FingerprintFFN** — learnable retrieval models
  from MassSpecGym, trained with cached fingerprints + InChIKeys (200×
  speedup) and a small `lr × hidden` grid; best config per model selected
  by formula val_hit_rate@1, used for both mass and formula test pickles.

In [1]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
from tqdm import tqdm

from massspecgym.utils import get_ci

tqdm.pandas()

seed = 0
random.seed(seed)
np.random.seed(seed)
pd.set_option('compute.use_numexpr', False)
pd.set_option('compute.use_bottleneck', False)

DIR_RESULTS = Path('/pfs/lustrep2/scratch/project_465002061/rbushuie/DreaMS-Mol_dev/MassSpecGym/data/test_results_v1.5/retrieval')
DATASET_PTH = Path('/pfs/lustrep2/scratch/project_465002061/rbushuie/DreaMS-Mol_dev/MassSpecGym/data/v1.5/MassSpecGym1.5.tsv')

In [2]:
def evaluate(dir_results, method_pkls, dataset_pth=None, metric_cols=None):
    """Per-spectrum bootstrap evaluation. ``method_pkls`` is a dict
    ``{method_label: pkl_filename}``. Mirrors evaluation.ipynb::evaluate but
    accepts an explicit method list and is robust to missing metric columns.

    test_mces@1 is reported in raw MCES distance units (smaller is better);
    hit-rates and MRR are reported as %.
    """
    np.random.seed(seed)
    if metric_cols is None:
        metric_cols = ['test_hit_rate@1', 'test_hit_rate@5', 'test_hit_rate@20',
                       'test_mrr', 'test_mces@1']

    gt_smiles = None
    if dataset_pth is not None:
        tsv = pd.read_csv(dataset_pth, sep='\t', usecols=['identifier', 'smiles'])
        gt_smiles = dict(zip(tsv['identifier'], tsv['smiles']))

    def _row_rr(row):
        if gt_smiles is None or 'sorted_candidate_smiles' not in row.index:
            return np.nan
        gt = gt_smiles.get(row['identifier'])
        if gt is None:
            return 0.0
        try:
            rank = list(row['sorted_candidate_smiles']).index(gt) + 1
            return 1.0 / rank
        except (ValueError, TypeError):
            return 0.0

    dfs = []
    for label, fn in method_pkls.items():
        path = dir_results / fn
        df_m = pd.read_pickle(path)
        df_m['method'] = label
        # Fill missing MRR from sorted_candidate_smiles + TSV lookup; else leave NaN.
        if 'test_mrr' not in df_m.columns:
            if 'sorted_candidate_smiles' in df_m.columns and gt_smiles is not None:
                df_m['test_mrr'] = df_m.apply(_row_rr, axis=1)
            else:
                df_m['test_mrr'] = np.nan
        dfs.append(df_m)
    df = pd.concat(dfs, ignore_index=True)

    # Render hit-rates / MRR as % to match evaluation.ipynb. test_mces@1 stays raw.
    for col in [c for c in df.columns if 'hit_rate' in c]:
        df[col] = df[col] * 100
    if 'test_mrr' in df.columns:
        df['test_mrr'] = df['test_mrr'] * 100

    cols_present = [c for c in metric_cols if c in df.columns]
    df_mean = df.groupby('method', sort=False)[cols_present].mean().round(2)

    def _ci_str(vals):
        v = pd.Series(vals).dropna().values
        if len(v) < 2:
            return '—'
        lo, hi = get_ci(v, confidence_level=0.999, n_resamples=20_000, seed=seed)
        return f'{lo:.2f}-{hi:.2f}'

    tqdm.pandas(desc='Bootstrapping per method', postfix=None)
    df_ci = df.groupby('method', sort=False)[cols_present].progress_apply(
        lambda dm: dm.apply(_ci_str, axis=0)
    )

    for c in cols_present:
        df_mean[c] = df_mean[c].astype(str) + ' (' + df_ci[c] + ')'

    # Preserve the input order of methods.
    df_mean = df_mean.reindex(list(method_pkls.keys()))
    return df_mean

## Mass-filtered candidate pool (S4+PubChem, ±10 ppm; stereo stripped)

In [3]:
mass_methods_all = {
    'random':         'random_mass_nostereo.pkl',
    'chirality':      'chirality_mass_nostereo_per_spectrum.pkl',
    'chemberta':      'chemberta_mass_nostereo.pkl',
    # Best-of-grid per architecture, selected by formula val_hit_rate@1 over
    # lr ∈ {1e-3, 1e-4} × hidden ∈ {256, 512} (2×2 grid per model).
    'deepsets':       'grid_deepsets_mass_lr1e-3_h256_nostereo.pkl',
    'deepsets+FF':    'grid_deepsets_ff_mass_lr1e-4_h512_nostereo.pkl',
    'fingerprintFFN': 'grid_fp_ffn_mass_lr1e-3_h512_nostereo.pkl',
}
mass_methods = {k: v for k, v in mass_methods_all.items() if (DIR_RESULTS / v).exists()}
print('mass methods available:', list(mass_methods))
df_mass = evaluate(DIR_RESULTS, mass_methods, dataset_pth=DATASET_PTH)
display(df_mass)

mass methods available:

['random', 'chirality', 'chemberta', 'deepsets', 'deepsets+FF', 'fingerprintFFN']

Bootstrapping per method:   0%|          | 0/6 [00:00<?, ?it/s]

Bootstrapping per method:  33%|███▎      | 2/6 [01:14<02:28, 37.17s/it]

Bootstrapping per method:  50%|█████     | 3/6 [02:14<02:19, 46.62s/it]

Bootstrapping per method:  67%|██████▋   | 4/6 [03:26<01:52, 56.36s/it]

Bootstrapping per method:  83%|████████▎ | 5/6 [04:38<01:01, 61.78s/it]

Bootstrapping per method: 100%|██████████| 6/6 [05:53<00:00, 65.99s/it]

Bootstrapping per method: 100%|██████████| 6/6 [07:07<00:00, 71.26s/it]

,test_hit_rate@1,test_hit_rate@5,test_hit_rate@20,test_mrr,test_mces@1
method,,,,,
random,0.45 (0.42-0.48),2.53 (2.39-2.67),9.44 (9.07-9.86),2.56 (2.48-2.66),27.62 (27.29-27.95)
chirality,0.5 (0.47-0.53),2.6 (2.46-2.76),9.57 (9.20-9.99),nan (—),27.47 (27.14-27.80)
chemberta,1.69 (1.38-2.02),5.38 (4.84-5.94),12.04 (11.26-12.91),4.16 (3.83-4.51),27.31 (27.00-27.63)
deepsets,0.7 (0.52-0.93),2.66 (2.28-3.09),8.61 (7.92-9.30),2.65 (2.43-2.92),24.89 (24.63-25.16)
deepsets+FF,5.76 (5.22-6.35),13.53 (12.71-14.42),25.82 (24.72-26.90),10.47 (9.87-11.07),21.37 (21.11-21.64)
fingerprintFFN,5.57 (5.03-6.17),10.45 (9.70-11.24),19.25 (18.28-20.20),8.89 (8.33-9.50),23.0 (22.71-23.28)


## Formula-filtered candidate pool (S4+PubChem+Molpher; stereo stripped)

In [4]:
formula_methods_all = {
    'random':         'random_formula_nostereo.pkl',
    'chirality':      'chirality_formula_nostereo_per_spectrum.pkl',
    'chemberta':      'chemberta_formula_nostereo.pkl',
    # Best-of-grid per architecture, selected by formula val_hit_rate@1 over
    # lr ∈ {1e-3, 1e-4} × hidden ∈ {256, 512} (2×2 grid per model).
    'deepsets':       'grid_deepsets_formula_lr1e-3_h256_nostereo.pkl',
    'deepsets+FF':    'grid_deepsets_ff_formula_lr1e-4_h512_nostereo.pkl',
    'fingerprintFFN': 'grid_fp_ffn_formula_lr1e-3_h512_nostereo.pkl',
}
formula_methods = {k: v for k, v in formula_methods_all.items() if (DIR_RESULTS / v).exists()}
print('formula methods available:', list(formula_methods))
df_formula = evaluate(DIR_RESULTS, formula_methods, dataset_pth=DATASET_PTH)
display(df_formula)

formula methods available:

['random', 'chirality', 'chemberta', 'deepsets', 'deepsets+FF', 'fingerprintFFN']

Bootstrapping per method:   0%|          | 0/6 [00:00<?, ?it/s]

Bootstrapping per method:  33%|███▎      | 2/6 [01:16<02:33, 38.26s/it]

Bootstrapping per method:  50%|█████     | 3/6 [02:14<02:19, 46.63s/it]

Bootstrapping per method:  67%|██████▋   | 4/6 [03:30<01:54, 57.28s/it]

Bootstrapping per method:  83%|████████▎ | 5/6 [04:42<01:02, 62.59s/it]

Bootstrapping per method: 100%|██████████| 6/6 [05:54<00:00, 65.82s/it]

Bootstrapping per method: 100%|██████████| 6/6 [07:05<00:00, 70.98s/it]

,test_hit_rate@1,test_hit_rate@5,test_hit_rate@20,test_mrr,test_mces@1
method,,,,,
random,1.89 (1.81-1.97),9.28 (8.93-9.65),28.5 (27.65-29.41),6.96 (6.75-7.17),14.89 (14.76-15.02)
chirality,1.87 (1.80-1.95),9.1 (8.75-9.47),28.84 (27.99-29.75),nan (—),14.62 (14.49-14.76)
chemberta,2.12 (1.79-2.51),10.28 (9.54-11.04),33.28 (32.11-34.54),7.62 (7.24-8.04),14.86 (14.73-14.98)
deepsets,3.1 (2.68-3.54),9.64 (8.90-10.34),27.0 (25.94-28.11),7.86 (7.41-8.31),14.87 (14.73-15.00)
deepsets+FF,5.58 (5.05-6.17),17.73 (16.81-18.68),37.7 (36.57-38.91),12.56 (11.98-13.17),13.87 (13.72-14.00)
fingerprintFFN,5.99 (5.45-6.61),16.97 (16.08-17.93),36.39 (35.19-37.63),12.51 (11.93-13.14),14.12 (13.96-14.27)


## Notes

- **Random** is the per-spectrum theoretical baseline: each spectrum has
  its own candidate-list size `N_i`; per-spectrum hit@k is drawn from
  Bernoulli(k / N_i), averaged over 100 draws per query.
- **Chirality** = score each candidate by RDKit chiral-atom count, fitted
  direction (asc/desc) from MSG-train, random tie-breaking. After
  stripping stereo from both queries and candidates the chir_count is 0
  for almost every molecule — this baseline collapses to near-random.
- **ChemBERTa-77M-MLM binary classifier** trained per candidate variant
  on (query=positive, candidates=negative) pairs from MSG-train, scored
  at test time as P(class=1). See `scripts/train_eval_chemberta_binary.py`.
- **DeepSets / DeepSets+FF / FingerprintFFN** — spectrum-conditioned
  retrieval baselines from MassSpecGym, trained with
  `--log_only_loss_at_stages=train` (skips per-train-batch MCES@1 +
  retrieval eval; val/test metrics still computed at epoch end).
  A small 2×2 hyperparameter grid (`lr ∈ {1e-3, 1e-4}` × `hidden ∈ {256, 512}`)
  was run per model with `EarlyStopping(monitor='val_hit_rate@1', patience=3)`.
  Best config per model was selected by formula val_hit_rate@1; both formula
  and mass test pickles come from that selected checkpoint.
  Training used cached Morgan FPs (4096-bit, bit-packed HDF5) and cached
  2D-InChIKey labels — yielding ~200× speedup vs uncached.
- **test_mces@1**: MyopicMCES distance between query SMILES and top-1
  predicted candidate (smaller = structurally closer). For random and
  chirality the top-1 was re-derived from the candidate JSON; for
  learnable models it comes from `sorted_candidate_smiles[0]`. Computed
  by `scripts/add_mces_at_1.py` on the per-spectrum pkls.

**Headline.**
- After stereo-stripping + 2D-InChIKey dedup, **all spectrum-free methods
  (chirality, ChemBERTa, even bare DeepSets) sit at or near random on
  formula**; ChemBERTa is ~4× random hit@1 on mass but its **MCES@1 is
  essentially random** (27.3 vs 27.6) — when wrong, its top-1 is not
  structurally close to the GT.
- **Spectrum-conditioned learnable methods clear the bar on both metrics**:
  - hit@1: DS+FF and FP-FFN reach **5.6–6.0 % on formula (3× random) and
    5.6–5.8 % on mass (12× random)**.
  - MCES@1: DS+FF best (**21.37 mass / 13.87 formula** vs random 27.62 / 14.89),
    FP-FFN close second. Even when these models pick wrong, the top-1 is
    structurally much closer to the GT.
- Bare DeepSets without Fourier features stays close to random on hit@1
  but its MCES@1 already drops (mass 24.89 < random 27.62), suggesting
  the model has learned **some** spectrum→structure information that
  hit@1 alone underplays.